# Mini GR00T + SONIC: BONES curriculum on Colab

This notebook mounts Google Drive, clones `YigitGunduc/robot`, extracts a SONIC-filtered BONES-SEED candidate pool, builds an auditable five-stage kinematic curriculum, runs a CUDA/MuJoCo-Warp smoke test, and dynamically promotes the body controller using held-out metrics. The optional final section collects replay and trains the text-to-token flow model.

Before starting, select **Runtime → Change runtime type → NVIDIA GPU**. Push the notebook and associated code changes to the repository before cloning from a fresh Colab runtime.

In [ ]:
from pathlib import Path

# Paths supplied by the Drive layout in the prompt.
DRIVE_BONES = Path('/content/drive/MyDrive/Datasets/bones-seed')
DRIVE_WORK = Path('/content/drive/MyDrive/mini_groot_sonic_colab')
REPO_URL = 'https://github.com/YigitGunduc/robot.git'  # HTTPS works in Colab without your Mac SSH key.
REPO_BRANCH = 'master'
REPO_DIR = Path('/content/robot')
MENAGERIE_DIR = Path('/content/mujoco_menagerie')

SEED = 0
CURRICULUM_CANDIDATES = 256  # preprocess once, then select the easiest safe 64
CURRICULUM_STAGE_SIZES = (8, 20, 32, 48, 64)
NUM_ENVS = 64             # raise to 128 only after the smoke test is stable
EVALUATION_CHUNK_ITERATIONS = 100
MINIMUM_STAGE_ITERATIONS = 1000
MAXIMUM_STAGE_ITERATIONS = 20000
PROMOTION_PATIENCE = 2
VISUAL_ROLLOUTS = 3       # held-out side-by-side policy/reference MP4 files
VIDEO_MAX_STEPS = 500     # 10 seconds at the 50 Hz controller rate
REPLAY_EPISODES = 48
FLOW_EPOCHS = 15

COPY_ARCHIVE_TO_LOCAL = False  # True is faster if the archive and extracted files fit /content.
FORCE_RAW_SELECTION = False
FORCE_PREPROCESS = False
FORCE_CURRICULUM = False
RESUME_EXISTING_RUNS = True
RUN_BODY_TRAINING = True
RUN_REPLAY_AND_FLOW = False    # Enable only after held-out body tracking is acceptable.

BALANCE_KEYWORDS = ('stand', 'standing', 'idle')
WALK_KEYWORDS = ('walk', 'walking')
TURN_KEYWORDS = ('turn', 'turning', 'pivot')
RUN_KEYWORDS = ('jog', 'jogging', 'run', 'running', 'sprint')
CURRICULUM_KEYWORD_GROUPS = (BALANCE_KEYWORDS, WALK_KEYWORDS, TURN_KEYWORDS, RUN_KEYWORDS)
EXPANDED_KEYWORDS = tuple(dict.fromkeys(term for group in CURRICULUM_KEYWORD_GROUPS for term in group))
# SONIC's upstream filename denylist is imported from the repository after installation.


In [ ]:
import shutil
import subprocess
import sys

from google.colab import drive

drive.mount('/content/drive')
assert DRIVE_BONES.is_dir(), f'BONES directory not found: {DRIVE_BONES}'
assert (DRIVE_BONES / 'metadata').is_dir(), 'BONES metadata directory is missing'
archives = [DRIVE_BONES / 'g1.tar.zst', DRIVE_BONES / 'g1.tar.gz']
assert any(path.exists() for path in archives), 'Neither g1.tar.zst nor g1.tar.gz was found'
DRIVE_WORK.mkdir(parents=True, exist_ok=True)
subprocess.run(['nvidia-smi'], check=True)


## Clone the code and install dependencies

For a private GitHub repository, replace `REPO_URL` with an authenticated HTTPS URL or configure a Colab secret. Do not paste a long-lived token into a saved notebook.

In [ ]:
if (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f'{REPO_DIR} exists but is not a Git checkout; remove or rename it')
else:
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)
    ], check=True)

subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'zstd'], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO_DIR}[all]', 'pyarrow'
], check=True)

# Editable installs are exposed through a .pth file that this already-running
# Colab kernel will not re-read until restart. Add src now so imports work in
# the very next cell while new subprocesses continue using the installation.
import importlib

source_root = str(REPO_DIR / 'src')
if source_root not in sys.path:
    sys.path.insert(0, source_root)
importlib.invalidate_caches()
import mini_groot_sonic

print('Imported mini_groot_sonic from', mini_groot_sonic.__file__)
subprocess.run(['git', '-C', str(REPO_DIR), 'log', '-1', '--oneline'], check=True)


## Fetch and validate the 29-DOF G1 MJCF

The project intentionally does not duplicate robot meshes. This cell obtains the maintained Unitree G1 model from MuJoCo Menagerie and validates the exact joint/actuator contract before preprocessing.

In [ ]:
MENAGERIE_URL = 'https://github.com/google-deepmind/mujoco_menagerie.git'
if not (MENAGERIE_DIR / '.git').is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', MENAGERIE_URL, str(MENAGERIE_DIR)], check=True)
MJCF = MENAGERIE_DIR / 'unitree_g1' / 'scene.xml'
assert MJCF.exists(), f'G1 scene not found: {MJCF}'

# Menagerie attaches the head mesh directly to torso_link, so keep the SONIC v3
# torso proxy explicit in the generated Colab configuration.
import yaml

COLAB_CONFIG = Path('/content/mini_groot_sonic_colab.yaml')
config_values = yaml.safe_load((REPO_DIR / 'configs/default.yaml').read_text())
config_values.setdefault('sim', {})['keypoint_body_names'] = [
    'torso_link', 'left_wrist_yaw_link', 'right_wrist_yaw_link',
    'left_ankle_roll_link', 'right_ankle_roll_link',
]
COLAB_CONFIG.write_text(yaml.safe_dump(config_values, sort_keys=False))

import mujoco

from mini_groot_sonic.config import load_project_config
from mini_groot_sonic.sim.g1_control import sonic_g1_control_profile
from mini_groot_sonic.sim.g1_mapping import G1ModelMap

cfg = load_project_config(COLAB_CONFIG)
model = mujoco.MjModel.from_xml_path(str(MJCF))
mapping = G1ModelMap.from_mjmodel(
    model, cfg.sim.root_body_name, tuple(cfg.sim.keypoint_body_names)
)
kind = 'position' if mapping.actuator_is_position.all() else 'motor/PD torque'
print(f'MJCF={MJCF}')
print(f'nq={model.nq}, nv={model.nv}, actuated_joints={len(mapping.joint_names)}, actuator_mode={kind}')
profile = sonic_g1_control_profile(mapping.joint_names)
print(mapping.joint_names)
print(f'SONIC action scales: {profile.action_scale.min():.4f}..{profile.action_scale.max():.4f} rad')
print(f'SONIC position gains: {profile.stiffness.min():.1f}..{profile.stiffness.max():.1f}')


## Select and extract only basic BONES motions

This applies SONIC's offline denylist and locomotion allowlist to archive paths/filenames, then extracts a 256-motion candidate pool. Structured BONES fields and measured G1 kinematics later reject interactions and rank difficulty. Whole source motions are kept, and the exact candidate selection is cached on Drive.

In [ ]:
import json
import os
import random

from mini_groot_sonic.data.bones import (
    SONIC_DEFAULT_FILTER_KEYWORDS,
    sonic_filename_allowed,
)

SONIC_FILTER_KEYWORDS = SONIC_DEFAULT_FILTER_KEYWORDS
selection_name = f'dynamic_curriculum_v4_seed{SEED}_n{CURRICULUM_CANDIDATES}'
run_name = f'{selection_name}_sonic_stack_v4'  # Never resume legacy reference scaling.
RAW_CACHE = DRIVE_WORK / 'cache' / selection_name
RAW_SELECTED = Path('/content/bones_selected_csv')
MANIFEST_PATH = RAW_CACHE / 'selection.json'

if FORCE_RAW_SELECTION and RAW_CACHE.exists():
    shutil.rmtree(RAW_CACHE)
if RAW_SELECTED.exists():
    shutil.rmtree(RAW_SELECTED)
RAW_SELECTED.mkdir(parents=True)

if MANIFEST_PATH.exists() and list(RAW_CACHE.glob('*.csv')):
    print('Using cached SONIC-style filename selection from Drive')
    shutil.copytree(RAW_CACHE, RAW_SELECTED, dirs_exist_ok=True, ignore=shutil.ignore_patterns('selection.json'))
    manifest = json.loads(MANIFEST_PATH.read_text())
else:
    archive = next(path for path in archives if path.exists())
    if COPY_ARCHIVE_TO_LOCAL:
        local_archive = Path('/content') / archive.name
        if not local_archive.exists() or local_archive.stat().st_size != archive.stat().st_size:
            print(f'Copying {archive.name} to local scratch...')
            shutil.copy2(archive, local_archive)
        archive = local_archive

    list_cmd = ['tar']
    if archive.suffix == '.zst':
        list_cmd += ['--zstd']
    list_cmd += ['-tf', str(archive)]
    print(f'Scanning archive members from {archive} ...')
    listing = subprocess.run(list_cmd, check=True, capture_output=True, text=True).stdout.splitlines()
    csv_members = [name for name in listing if name.lower().endswith('.csv')]
    member_by_stem = {}
    for member in csv_members:
        member_by_stem.setdefault(Path(member).stem, member)
    print(f'Archive contains {len(csv_members):,} CSV members and {len(member_by_stem):,} unique stems')

    def candidates_for(keywords):
        return [
            stem for stem, member in member_by_stem.items()
            if sonic_filename_allowed(member, keywords, SONIC_FILTER_KEYWORDS)
        ]

    selected_stems = []
    quota = CURRICULUM_CANDIDATES // len(CURRICULUM_KEYWORD_GROUPS)
    for group_index, keywords in enumerate(CURRICULUM_KEYWORD_GROUPS):
        group = sorted(set(candidates_for(keywords)) - set(selected_stems))
        random.Random(SEED + group_index).shuffle(group)
        group_target = quota if group_index < len(CURRICULUM_KEYWORD_GROUPS) - 1 else CURRICULUM_CANDIDATES - len(selected_stems)
        selected_stems.extend(group[:group_target])
    if len(selected_stems) < CURRICULUM_CANDIDATES:
        fallback = sorted(set(candidates_for(EXPANDED_KEYWORDS)) - set(selected_stems))
        random.Random(SEED + 100).shuffle(fallback)
        selected_stems.extend(fallback[:CURRICULUM_CANDIDATES - len(selected_stems)])
    assert len(selected_stems) == CURRICULUM_CANDIDATES, (
        f'Only {len(selected_stems)} filename candidates matched; requested {CURRICULUM_CANDIDATES}'
    )

    selected_members = [member_by_stem[stem] for stem in selected_stems]
    member_file = Path('/content/selected_bones_members.txt')
    member_file.write_text('\n'.join(selected_members) + '\n')
    extract_root = Path('/content/bones_extract')
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir()
    extract_cmd = ['tar']
    if archive.suffix == '.zst':
        extract_cmd += ['--zstd']
    extract_cmd += ['-xf', str(archive), '-C', str(extract_root), '-T', str(member_file)]
    print(f'Extracting only {len(selected_members)} selected CSV files...')
    subprocess.run(extract_cmd, check=True)
    for stem, member in zip(selected_stems, selected_members):
        relative = member.removeprefix('./')
        source = extract_root / relative
        assert source.exists(), f'Archive member was not extracted: {member}'
        shutil.copy2(source, RAW_SELECTED / f'{stem}.csv')

    manifest = {
        'seed': SEED, 'selection_method': 'sonic_filename_filter',
        'temporal_segments': False,
        'keyword_groups': CURRICULUM_KEYWORD_GROUPS, 'expanded_keywords': EXPANDED_KEYWORDS,
        'exclude_keywords': SONIC_FILTER_KEYWORDS,
        'candidate_stems': selected_stems,
    }
    RAW_CACHE.mkdir(parents=True, exist_ok=True)
    shutil.copytree(RAW_SELECTED, RAW_CACHE, dirs_exist_ok=True)
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))

def make_candidate_root(selected_stems):
    root = Path('/content/bones_curriculum_candidates')
    if root.exists():
        shutil.rmtree(root)
    csv_dir = root / 'g1' / 'csv' / 'selected'
    csv_dir.mkdir(parents=True)
    os.symlink(DRIVE_BONES / 'metadata', root / 'metadata', target_is_directory=True)
    for stem in selected_stems:
        source = RAW_SELECTED / f'{stem}.csv'
        target = csv_dir / source.name
        try:
            os.link(source, target)
        except OSError:
            shutil.copy2(source, target)
    return root

BONES_CANDIDATE_ROOT = make_candidate_root(manifest['candidate_stems'])
print(f"Raw curriculum candidates: {len(manifest['candidate_stems'])}")
print('First candidate IDs:', manifest['candidate_stems'][:10])


## Preprocess, audit, and build the curriculum

The candidate pool is preprocessed once and cached. The builder rejects mirrors, props, complex/non-locomotion records, and kinematic quality failures; ranks remaining motions by measured difficulty; then writes cumulative stage manifests plus `audit.csv`.

In [ ]:
import pandas as pd

PREPROCESS_CACHE = DRIVE_WORK / 'preprocessed' / selection_name / 'candidates'
CANDIDATE_DATA = Path('/content/mgsp_preprocessed/candidates')
if CANDIDATE_DATA.exists():
    shutil.rmtree(CANDIDATE_DATA)
if not FORCE_PREPROCESS and PREPROCESS_CACHE.exists() and list(PREPROCESS_CACHE.glob('*.npz')):
    print('Copying cached candidate preprocessing from Drive...')
    shutil.copytree(PREPROCESS_CACHE, CANDIDATE_DATA)
else:
    CANDIDATE_DATA.mkdir(parents=True)
    subprocess.run([
        sys.executable, '-m', 'mini_groot_sonic.tools.preprocess_bones',
        '--bones-root', str(BONES_CANDIDATE_ROOT), '--mjcf', str(MJCF),
        '--out', str(CANDIDATE_DATA), '--limit', str(CURRICULUM_CANDIDATES),
        '--seed', str(SEED), '--include-keywords', ','.join(EXPANDED_KEYWORDS),
        '--exclude-keywords', ','.join(SONIC_FILTER_KEYWORDS),
        '--no-temporal-segments', '--min-clip-seconds', '1.0',
    ], cwd=REPO_DIR, check=True)
    if PREPROCESS_CACHE.exists():
        shutil.rmtree(PREPROCESS_CACHE)
    shutil.copytree(CANDIDATE_DATA, PREPROCESS_CACHE)

CURRICULUM_DIR = DRIVE_WORK / 'curricula' / selection_name
CURRICULUM_MANIFEST = CURRICULUM_DIR / 'curriculum.json'
if FORCE_CURRICULUM and CURRICULUM_DIR.exists():
    shutil.rmtree(CURRICULUM_DIR)
if not CURRICULUM_MANIFEST.exists():
    subprocess.run([
        sys.executable, '-m', 'mini_groot_sonic.tools.build_curriculum',
        '--motions', str(CANDIDATE_DATA), '--out', str(CURRICULUM_DIR),
        '--stage-sizes', ','.join(map(str, CURRICULUM_STAGE_SIZES)), '--seed', str(SEED),
    ], cwd=REPO_DIR, check=True)
else:
    print('Using cached curriculum manifest from Drive')

curriculum_manifest = json.loads(CURRICULUM_MANIFEST.read_text())
for stage in curriculum_manifest['stages']:
    print(f"{stage['index']}: {stage['name']}: {len(stage['filenames'])} total, {len(stage['new_filenames'])} new")
audit = pd.read_csv(CURRICULUM_DIR / 'audit.csv')
print('Semantic decisions:')
print(audit['semantic_reason'].value_counts(dropna=False).to_string())
print('Quality decisions:')
print(audit['quality_reason'].value_counts(dropna=False).head(10).to_string())
print('Audit CSV:', CURRICULUM_DIR / 'audit.csv')


## Tests and target-GPU smoke test

Do not start PPO unless this cell passes. It catches dependency, MJCF, CUDA interop, reset, reference-shape, and actuator-boundary failures on the actual Colab GPU.

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=REPO_DIR, check=True)

import torch

from mini_groot_sonic.data.motion_bank import MotionBank
from mini_groot_sonic.sim.math_utils import quat_distance_angle
from mini_groot_sonic.sim.mjwarp_env import MJWarpG1VecEnv

assert torch.cuda.is_available(), 'CUDA is unavailable; choose a GPU Colab runtime'
cfg = load_project_config(COLAB_CONFIG)
cfg.sim.mjcf = MJCF
cfg.sim.device = 'cuda:0'
smoke_paths = [CANDIDATE_DATA / name for name in curriculum_manifest['stages'][0]['filenames'][:2]]
for randomized in (False, True):
    cfg.sim.enable_randomization = randomized
    bank = MotionBank(smoke_paths, cfg.sonic, cfg.sim.device)
    env = MJWarpG1VecEnv(cfg.sim, cfg.sonic, len(smoke_paths))
    ids = torch.arange(len(smoke_paths), device=cfg.sim.device)
    frames = torch.zeros(len(smoke_paths), dtype=torch.long, device=cfg.sim.device)
    ref = bank.current_reference(ids, frames)
    obs = env.reset(
        ref['root_pos'], ref['root_quat'], ref['joint_pos'],
        ref['root_linvel'], ref['root_angvel'], ref['joint_vel'],
    )
    obs = env.step(torch.zeros(len(smoke_paths), cfg.sonic.dof, device=cfg.sim.device))
    future = bank.future_reference(ids, frames, obs.root_quat)
    assert future.shape == (len(smoke_paths), cfg.sonic.future_frames, cfg.sonic.reference_frame_dim)
    assert torch.isfinite(obs.joint_pos).all() and torch.isfinite(obs.root_pos).all()
    print(f'CUDA/MJWarp smoke passed: randomized={randomized}, actuator={env.actuator_mode}, future={tuple(future.shape)}')
    if not randomized:
        stand_root = torch.zeros(len(smoke_paths), 3, device=cfg.sim.device)
        stand_root[:, 2] = 0.76
        stand_quat = torch.zeros(len(smoke_paths), 4, device=cfg.sim.device)
        stand_quat[:, 0] = 1.0
        stand_joint = env.default_joint_pos[None].expand(len(smoke_paths), -1)
        obs = env.reset(stand_root, stand_quat, stand_joint)
        zero_action = torch.zeros(len(smoke_paths), cfg.sonic.dof, device=cfg.sim.device)
        minimum_height = float(obs.root_pos[:, 2].min())
        for _ in range(25):  # short open-loop plant check at 50 Hz
            obs = env.step(zero_action)
            minimum_height = min(minimum_height, float(obs.root_pos[:, 2].min()))
        tilt = quat_distance_angle(obs.root_quat, stand_quat)
        assert minimum_height > 0.65 and float(tilt.max()) < 0.5, (
            f'Calibrated standing smoke failed: min_height={minimum_height:.3f}, max_tilt={float(tilt.max()):.3f}'
        )
        print(f'Half-second calibrated plant check passed: min_height={minimum_height:.3f}, max_tilt={float(tilt.max()):.3f}')
    del env, bank, obs, future
torch.cuda.empty_cache()


## Dynamically train and promote curriculum stages

Training evaluates every chunk and promotes only after all held-out gates pass twice consecutively. Stages are cumulative, so earlier motions remain available. Domain randomization starts at the turning stage. This SONIC-aligned control stack writes to a new `sonic_stack_v4` run and intentionally cannot resume older checkpoints whose reference scaling differs. If a stage exhausts its budget, training stops safely and the evaluation cells below still use its latest checkpoint.

In [ ]:
CURRICULUM_RUN = DRIVE_WORK / 'runs' / run_name / 'body_curriculum'
CURRICULUM_STATE = CURRICULUM_RUN / 'curriculum_state.json'
if not RESUME_EXISTING_RUNS and CURRICULUM_STATE.exists():
    raise RuntimeError(f'Existing curriculum state found at {CURRICULUM_STATE}; choose a new run_name')
if RUN_BODY_TRAINING:
    cmd = [
        sys.executable, '-u', '-m', 'mini_groot_sonic.tools.train_curriculum',
        '--manifest', str(CURRICULUM_MANIFEST), '--motions', str(CANDIDATE_DATA),
        '--config', str(COLAB_CONFIG), '--mjcf', str(MJCF), '--device', 'cuda:0',
        '--num-envs', str(NUM_ENVS), '--out', str(CURRICULUM_RUN),
        '--evaluation-chunk-iterations', str(EVALUATION_CHUNK_ITERATIONS),
        '--minimum-stage-iterations', str(MINIMUM_STAGE_ITERATIONS),
        '--maximum-stage-iterations', str(MAXIMUM_STAGE_ITERATIONS),
        '--promotion-patience', str(PROMOTION_PATIENCE),
        '--randomization-start-stage', '3', '--randomization',
    ]
    subprocess.run(cmd, cwd=REPO_DIR, check=True)
assert CURRICULUM_STATE.exists(), 'Curriculum did not create a state file'
training_state = json.loads(CURRICULUM_STATE.read_text())
print(json.dumps(training_state, indent=2))


## Inspect the current promotion decision

The state history records every numerical gate, failed condition, consecutive pass count, and checkpoint. A `stage_budget_exhausted` status means the model did not advance; increase the stage budget only after inspecting its numerical and visual results.

In [ ]:
last_gate = training_state['history'][-1] if training_state['history'] else None
print('Status:', training_state['status'])
print('Current stage:', training_state['current_stage'])
print('Latest checkpoint:', training_state['latest_checkpoint'])
if last_gate is not None:
    print(json.dumps(last_gate, indent=2))


## Reproduce the held-out split and evaluate

The evaluation directory reproduces the curriculum's permanently reserved actor/source groups and evaluates the latest checkpoint from the current (or final completed) stage.

In [ ]:
from IPython.display import Video, display

from mini_groot_sonic.data.curriculum import load_curriculum_manifest
from mini_groot_sonic.training.utils import split_curriculum_motion_paths

_, curriculum_stages = load_curriculum_manifest(CURRICULUM_MANIFEST)
all_stage_paths = [
    [CANDIDATE_DATA / filename for filename in stage.filenames]
    for stage in curriculum_stages
]
stage_splits = split_curriculum_motion_paths(all_stage_paths, 0.15, SEED)
ACTIVE_STAGE_INDEX = min(training_state['current_stage'], len(curriculum_stages) - 1)
active_paths = all_stage_paths[ACTIVE_STAGE_INDEX]
_, validation_paths = stage_splits[ACTIVE_STAGE_INDEX]
ACTIVE_STAGE_DATA = Path('/content/mgsp_active_curriculum_stage')
if ACTIVE_STAGE_DATA.exists():
    shutil.rmtree(ACTIVE_STAGE_DATA)
ACTIVE_STAGE_DATA.mkdir()
for path in active_paths:
    os.symlink(path, ACTIVE_STAGE_DATA / path.name)
HELD_OUT = Path('/content/mgsp_held_out')
if HELD_OUT.exists():
    shutil.rmtree(HELD_OUT)
HELD_OUT.mkdir()
for path in validation_paths:
    os.symlink(path, HELD_OUT / path.name)

BODY_CHECKPOINT = Path(training_state['latest_checkpoint'])
assert BODY_CHECKPOINT.exists(), f'Latest curriculum checkpoint is missing: {BODY_CHECKPOINT}'
subprocess.run([
    sys.executable, '-m', 'mini_groot_sonic.tools.eval_body',
    '--motions', str(HELD_OUT), '--config', str(COLAB_CONFIG),
    '--mjcf', str(MJCF), '--body', str(BODY_CHECKPOINT),
    '--device', 'cuda:0', '--max-motions', str(max(1, len(validation_paths))),
], cwd=REPO_DIR, check=True)

# Render multiple held-out rollouts side-by-side against their BONES reference.
# MP4 files and JSON metrics sidecars persist on Drive.

# Prefer representative easy walk/stand and faster run/turn filenames when the
# held-out partition contains them, then fill remaining slots deterministically.
visual_paths = []
for keywords in (('stand', 'idle'), ('walk',), ('run', 'jog'), ('turn',)):
    match = next((p for p in validation_paths if p not in visual_paths and any(k in p.stem.lower() for k in keywords)), None)
    if match is not None:
        visual_paths.append(match)
for path in validation_paths:
    if len(visual_paths) >= VISUAL_ROLLOUTS:
        break
    if path not in visual_paths:
        visual_paths.append(path)
visual_paths = visual_paths[:VISUAL_ROLLOUTS]
VISUAL_HELD_OUT = Path('/content/mgsp_visual_held_out')
if VISUAL_HELD_OUT.exists():
    shutil.rmtree(VISUAL_HELD_OUT)
VISUAL_HELD_OUT.mkdir()
for index, path in enumerate(visual_paths):
    os.symlink(path, VISUAL_HELD_OUT / f'{index:02d}_{path.name}')

VIDEO_DIR = DRIVE_WORK / 'videos' / run_name
VIDEO_DIR.mkdir(parents=True, exist_ok=True)
video_paths = []
for motion_index in range(len(visual_paths)):
    video_path = VIDEO_DIR / f'held_out_{motion_index:02d}.mp4'
    subprocess.run([
        sys.executable, '-m', 'mini_groot_sonic.tools.render_body',
        '--motions', str(VISUAL_HELD_OUT), '--config', str(COLAB_CONFIG),
        '--mjcf', str(MJCF), '--body', str(BODY_CHECKPOINT),
        '--device', 'cuda:0', '--motion-index', str(motion_index),
        '--max-steps', str(VIDEO_MAX_STEPS), '--out', str(video_path),
    ], cwd=REPO_DIR, check=True)
    video_paths.append(video_path)
    display(Video(str(video_path), embed=True, width=960))
print('Saved visual evaluations to:', VIDEO_DIR)


## Optional: replay collection and compact GR00T-style flow training

Leave `RUN_REPLAY_AND_FLOW=False` until body success/MPJPE is acceptable. Then change it to `True` in the configuration cell and run this cell. It collects causal token replay and trains the 1.76M-parameter text-conditioned flow model.

In [ ]:
REPLAY_DIR = DRIVE_WORK / 'replays' / run_name
FLOW_DIR = DRIVE_WORK / 'runs' / run_name / 'flow_text'
if RUN_REPLAY_AND_FLOW:
    subprocess.run([
        sys.executable, '-m', 'mini_groot_sonic.tools.collect_replay',
        '--motions', str(ACTIVE_STAGE_DATA), '--config', str(COLAB_CONFIG), '--mjcf', str(MJCF),
        '--checkpoint', str(BODY_CHECKPOINT), '--mode', 'policy',
        '--out', str(REPLAY_DIR), '--limit', str(REPLAY_EPISODES),
        '--seed', str(SEED), '--device', 'cuda:0',
    ], cwd=REPO_DIR, check=True)
    flow_cmd = [
        sys.executable, '-m', 'mini_groot_sonic.tools.train_flow',
        '--replays', str(REPLAY_DIR), '--config', str(COLAB_CONFIG), '--out', str(FLOW_DIR),
        '--device', 'cuda:0', '--epochs', str(FLOW_EPOCHS),
        '--batch-size', '64', '--workers', '2',
    ]
    if RESUME_EXISTING_RUNS:
        flow_checkpoints = sorted(FLOW_DIR.glob('flow_[0-9]*.pt'))
        if flow_checkpoints:
            flow_cmd += ['--resume', str(flow_checkpoints[-1])]
    subprocess.run(flow_cmd, cwd=REPO_DIR, check=True)
else:
    print('Skipped replay/flow. Enable RUN_REPLAY_AND_FLOW after body validation passes.')


## What to inspect before scaling

1. `curriculum_state.json` should show promotion gates passing before stages advance.
2. Root error, MPJPE, undesired contacts, and action rate should trend downward.
3. FSQ occupancy should not collapse near zero; saturation should remain modest.
4. Only after the final locomotion stage is stable should you increase environments/motions or enable the upper flow model.
5. Do not add flips, jumps, stairs, or arbitrary target perturbations until basic tracking is robust.